# SupplySight AI

## Enterprise Procurement Decision Intelligence Platform

# Notebook 01G

# Country Holiday ETL

---

## Objective

The objective of this notebook is to construct an enterprise-grade historical
Country Holiday Dimension for the SupplySight AI Procurement Decision
Intelligence Platform.

Public holidays influence procurement activities by affecting supplier
availability, manufacturing schedules, logistics operations and transportation
lead times. Incorporating holiday information enables procurement decisions to
consider business calendar effects alongside internal ERP transactions.

Instead of relying on external holiday APIs with limited country coverage and
historical support, this notebook generates historical public holidays using
the Python **holidays** library together with the standardized Enterprise
Country Dimension developed in Notebook 01E.

The workflow includes:

- Loading the Enterprise Country Dimension
- Preparing standardized country holiday requests
- Generating historical public holidays
- Creating holiday and weekend indicators
- Validating dataset completeness
- Exporting the Enterprise Holiday Dimension

The resulting dataset will later be integrated with procurement transactions
using standardized country and date keys within the Enterprise Data
Integration Pipeline (Notebook 04A).

---

## Expected Output

### Enterprise Holiday Dimension

The generated dataset contains:

- Observation Date
- ERP Country
- ISO Country Code
- Holiday Name
- Holiday Indicator
- Weekend Indicator

The Holiday Dimension provides the business calendar context required for
enterprise procurement analytics, demand forecasting and supplier
performance analysis.

## Import Libraries

In [25]:
%pip install holidays

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [26]:
# ============================================================
# Import Libraries
# ============================================================

import pandas as pd
import holidays

from pathlib import Path

## Project Configuration

In [27]:
# ============================================================
# Configuration
# ============================================================

PROJECT_PATH = Path(
    r"C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform"
)

PROCESSED = PROJECT_PATH / "Datasets" / "Processed Datasets"

RAW = PROJECT_PATH / "Datasets" / "Raw Datasets"

EXTERNAL = PROJECT_PATH / "Datasets" / "External Datasets"

print("Project Paths Configured Successfully")

Project Paths Configured Successfully


## Load Country Dimension

In [43]:
# ============================================================
# Load Enterprise Country Dimension
# ============================================================

dim_country = pd.read_csv(
    PROCESSED / "dim_country.csv",
    dtype={"iso2": str},
    keep_default_na=False
)

print("="*70)
print("COUNTRY DIMENSION")
print("="*70)

print(f"Rows    : {dim_country.shape[0]}")
print(f"Columns : {dim_country.shape[1]}")

COUNTRY DIMENSION
Rows    : 164
Columns : 14


## Manual Master Data Corrections

In [44]:
# ============================================================
# Manual Master Data Corrections
# ============================================================

dim_country.loc[
    dim_country["erp_country"]=="Namibia",
    ["iso2","iso3"]
] = ["NA","NAM"]

print("="*70)
print("MASTER DATA CORRECTIONS")
print("="*70)

print(
    dim_country[
        dim_country["erp_country"]=="Namibia"
    ]
)

MASTER DATA CORRECTIONS
     country_key erp_country country_name iso2 iso3  region        subregion  \
103          104     Namibia      Namibia   NA  NAM  Africa  Southern Africa   

    continent   capital weather_reference_city        languages  geoname_id  \
103    Africa  Windhoek               Windhoek  en,af,de,hz,naq   3355338.0   

                     created_at            data_source  
103  2026-07-15 16:55:19.515053  DataHub Country Codes  


## Prepare Holiday Request Table

In [45]:
# ============================================================
# Prepare Holiday Request Table
# ============================================================

holiday_requests = dim_country[
    [
        "iso2",
        "country_name",
        "erp_country"
    ]
].copy()

print("="*70)
print("HOLIDAY REQUEST TABLE")
print("="*70)

print(f"Countries : {len(holiday_requests)}")

display(holiday_requests.head())

HOLIDAY REQUEST TABLE
Countries : 164


,iso2,country_name,erp_country
0,AF,Afghanistan,Afganistán
1,AL,Albania,Albania
2,DE,Germany,Alemania
3,AO,Angola,Angola
4,SA,Saudi Arabia,Arabia Saudí


In [46]:
holiday_requests[
    holiday_requests["erp_country"] == "Namibia"
]

,iso2,country_name,erp_country
103,NA,Namibia,Namibia


## Validate Holiday Library

In [47]:
# ============================================================
# Holiday Library Validation
# ============================================================

supported = []
unsupported = []

for _, row in holiday_requests.iterrows():

    try:

        holidays.country_holidays(
            row["iso2"],
            years=[2015]
        )

        supported.append(row)

    except Exception:

        unsupported.append(row)

supported = pd.DataFrame(supported)
unsupported = pd.DataFrame(unsupported)

print("="*70)
print("HOLIDAY LIBRARY VALIDATION")
print("="*70)

print(f"Supported   : {len(supported)}")
print(f"Unsupported : {len(unsupported)}")

display(unsupported)

HOLIDAY LIBRARY VALIDATION
Supported   : 164
Unsupported : 0


""


## Generate Holiday Event Table

In [48]:
# ============================================================
# Generate Holiday Events
# ============================================================

holiday_records = []

years = [2015,2016,2017,2018]

for _, row in supported.iterrows():

    calendar = holidays.country_holidays(
        row["iso2"],
        years=years,
        language="en"
    )

    for holiday_date, holiday_name in calendar.items():

        holiday_records.append({

            "date":pd.to_datetime(holiday_date),

            "iso2":row["iso2"],

            "country_name":row["country_name"],

            "erp_country":row["erp_country"],

            "holiday_name":holiday_name,

            "is_holiday":1

        })

holiday_events = pd.DataFrame(holiday_records)

print("="*70)
print("HOLIDAY EVENT TABLE")
print("="*70)

print(f"Rows : {holiday_events.shape[0]}")

HOLIDAY EVENT TABLE
Rows : 9354


## Build Enterprise Calendar

In [49]:
# ============================================================
# Enterprise Calendar
# ============================================================

calendar = pd.DataFrame({

    "date":pd.date_range(
        "2015-01-01",
        "2018-12-31",
        freq="D"
    )

})

holiday_calendar = holiday_requests.merge(
    calendar,
    how="cross"
)

print("="*70)
print("ENTERPRISE HOLIDAY CALENDAR")
print("="*70)

print(f"Rows : {holiday_calendar.shape[0]:,}")

ENTERPRISE HOLIDAY CALENDAR
Rows : 239,604


## Merge Holiday Events

In [50]:
# ============================================================
# Merge Holiday Events
# ============================================================

dim_holiday = holiday_calendar.merge(

    holiday_events[
        [
            "date",
            "iso2",
            "holiday_name",
            "is_holiday"
        ]
    ],

    on=[
        "date",
        "iso2"
    ],

    how="left",

    validate="one_to_one"

)

## Holiday Indicators

In [51]:
# ============================================================
# Holiday Indicators
# ============================================================

dim_holiday["is_holiday"] = (
    dim_holiday["is_holiday"]
    .fillna(0)
    .astype(int)
)

dim_holiday["holiday_name"] = (
    dim_holiday["holiday_name"]
    .fillna("No Public Holiday")
)

## Weekend Flag

In [52]:
# ============================================================
# Weekend Indicator
# ============================================================

dim_holiday["is_weekend"] = (

    dim_holiday["date"]
    .dt.dayofweek
    .isin([5,6])

).astype(int)

## Holiday Source Availability

In [53]:
# ============================================================
# Holiday Source Availability
# ============================================================

dim_holiday["holiday_data_available"] = (
    ~dim_holiday["iso2"].isin(["NA"])
).astype(int)

## Enterprise Validation

In [54]:
# ============================================================
# Enterprise Validation
# ============================================================

print("="*70)
print("FINAL ENTERPRISE HOLIDAY DIMENSION")
print("="*70)

print(f"Rows      : {dim_holiday.shape[0]:,}")
print(f"Columns   : {dim_holiday.shape[1]}")

print("\nDuplicate Keys")

print(
    dim_holiday
    .duplicated(["date","iso2"])
    .sum()
)

print("\nMissing Values")

print(
    dim_holiday
    .isnull()
    .sum()
)

print("\nHoliday Records")

print(
    dim_holiday["is_holiday"]
    .value_counts()
)

print("\nWeekend Records")

print(
    dim_holiday["is_weekend"]
    .value_counts()
)

print("\nHoliday Source Availability")

print(
    dim_holiday["holiday_data_available"]
    .value_counts()
)

FINAL ENTERPRISE HOLIDAY DIMENSION
Rows      : 239,604
Columns   : 8

Duplicate Keys
0

Missing Values
iso2                      0
country_name              0
erp_country               0
date                      0
holiday_name              0
is_holiday                0
is_weekend                0
holiday_data_available    0
dtype: int64

Holiday Records
is_holiday
0    230250
1      9354
Name: count, dtype: int64

Weekend Records
is_weekend
0    171052
1     68552
Name: count, dtype: int64

Holiday Source Availability
holiday_data_available
1    238143
0      1461
Name: count, dtype: int64


In [55]:
dim_holiday.rename(
    columns={
        "holiday_data_available":"holiday_source_available"
    },
    inplace=True
)

## Final Column Order

In [56]:
dim_holiday = dim_holiday[
    [
        "date",
        "iso2",
        "country_name",
        "erp_country",
        "holiday_name",
        "is_holiday",
        "is_weekend",
        "holiday_source_available"
    ]
]

## Export

In [57]:
# ============================================================
# Export Holiday Dimension
# ============================================================

dim_holiday.to_csv(
    PROCESSED / "dim_holiday.csv",
    index=False
)

# Notebook Summary

This notebook developed the **Enterprise Holiday Dimension** for the SupplySight AI
Enterprise Procurement Decision Intelligence Platform.

Using the standardized Enterprise Country Dimension created in Notebook **01E**,
historical public holidays were generated for ERP countries covering the period
from **1 January 2015** to **31 December 2018**.

Rather than storing only holiday events, this notebook constructed a complete
**Country-Date Holiday Dimension**, ensuring that every ERP country has one record
for every day within the analysis period. Public holiday indicators, weekend
indicators, and holiday source availability flags were incorporated to create a
robust enterprise calendar suitable for downstream analytics.

The Holiday Dimension follows the same Country-Date granularity as the Enterprise
Weather Dimension, enabling consistent and efficient integration with ERP
procurement transactions during the Enterprise Data Integration stage.

---

## Major Activities

- Loaded the Enterprise Country Dimension
- Applied master data corrections for standardized ISO country codes
- Prepared standardized holiday request tables
- Validated holiday library compatibility across ERP countries
- Generated historical public holiday events (2015–2018)
- Built the complete Enterprise Country-Date Holiday Calendar
- Merged holiday events into the enterprise calendar
- Created holiday indicators and standardized holiday names
- Generated weekend indicators
- Created holiday source availability flags
- Validated data quality, completeness, and key uniqueness
- Exported the Enterprise Holiday Dimension

---

## Output Dataset

```
dim_holiday.csv
```

---

## Final Dataset Statistics

| Metric | Value |
|--------|------:|
| Countries | 164 |
| Analysis Period | 2015–2018 |
| Calendar Days | 1,461 |
| Total Records | 239,604 |
| Holiday Records | 9,354 |
| Weekend Records | 68,552 |
| Duplicate Country-Date Keys | 0 |
| Missing Values | 0 |

---

## Business Value

The Enterprise Holiday Dimension enriches procurement transactions with
business calendar intelligence by identifying public holidays and weekends
across all ERP countries.

This additional context supports more accurate analysis of supplier performance,
shipping delays, procurement demand patterns, lead-time variability, and
operational disruptions influenced by national holidays and non-working days.

By combining standardized country information with historical business calendars,
the dimension provides a reliable foundation for enterprise reporting,
predictive analytics, and procurement decision intelligence.

---

## Next Step

The next notebook, **Notebook 02A – Enterprise Data Warehouse Design**, will
organize the ERP fact data and all processed external dimensions into a
Kimball-style enterprise star schema. The standardized dimensions developed in
the ETL phase—including Commodity, Fuel Price, Exchange Rate, Country, Weather,
and Holiday—will serve as the foundation for the Enterprise Data Warehouse and
subsequent data integration pipeline.